# Session 4 Lab — From Agent Map to Production Deployment

**AI Agents Workshop — Master of Quantitative Economics**

This notebook is an offline, API-free lab. Students design and test a production-minded agent system around:

1. Agent/system mapping
2. MCP-style tool interfaces
3. Spec-driven deployment
4. Guardrails and human review gates
5. Cost and latency control
6. Evaluation and observability

**Scenario:** build a research assistant that answers macro/market questions using a small internal dataset and controlled tools. The goal is not to build the most powerful agent; the goal is to make deployment decisions explicit, testable, and auditable.

## Learning objectives

By the end of this lab, students should be able to:

- Convert a high-level agent idea into a production map: users, tasks, data, tools, policies, risks, and evaluation metrics.
- Explain MCP as a standardized interface between an agent client and external servers/tools.
- Write a deployment specification that separates behavior, tools, policies, costs, and evaluation criteria.
- Implement guardrails for input validation, tool authorization, output validation, and human escalation.
- Measure approximate token cost and route tasks between cheap and expensive models.
- Produce an observability trace that can be inspected after the run.

In [ ]:
# Standard library only + pydantic if available.
# If pydantic is not installed, the notebook falls back to dataclasses.

from __future__ import annotations
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Literal, Callable
import json, re, time, math, statistics, uuid

try:
    from pydantic import BaseModel, Field, ValidationError
    PYDANTIC_AVAILABLE = True
except Exception:
    PYDANTIC_AVAILABLE = False

print('Pydantic available:', PYDANTIC_AVAILABLE)

## 1. Start with the production map

A production agent should be mapped before it is coded. For this session, use six layers:

| Layer | Questions |
|---|---|
| User + task | Who uses it? What decision does it support? |
| Data/context | What sources are allowed? What is stale, private, or risky? |
| Tools/actions | What can the agent read, calculate, write, or change? |
| Policy/guardrails | What must be blocked, escalated, redacted, or validated? |
| Runtime/cost | What model is used? What latency/cost budget applies? |
| Evaluation/observability | How do we know it worked? What trace is stored? |

### Exercise 1
Fill the map for a macro research assistant used by a portfolio strategy team.

In [ ]:
agent_map = {
    "user_task": {
        "primary_user": "macro/quant research analyst",
        "decision_supported": "produce a short evidence-backed macro note",
        "failure_cost": "medium: wrong evidence can mislead investment discussion"
    },
    "data_context": {
        "allowed_sources": ["internal_macro_table", "approved_research_notes"],
        "disallowed_sources": ["private client data", "unverified internet claims"],
        "freshness_requirement": "data timestamp must be shown"
    },
    "tools_actions": {
        "read_tools": ["search_macro_data", "retrieve_policy_note"],
        "compute_tools": ["calculate_summary_stats"],
        "write_tools": [],
        "external_side_effects": "none"
    },
    "policy_guardrails": {
        "input_blocks": ["requests for trades as personalized advice"],
        "tool_permissions": {"search_macro_data": "allow", "delete_data": "deny"},
        "output_requirements": ["cite evidence", "include uncertainty", "no unsupported recommendation"]
    },
    "runtime_cost": {
        "default_model": "small/cheap model for routing and extraction",
        "escalation_model": "larger model for synthesis only",
        "max_usd_per_run": 0.05
    },
    "evaluation_observability": {
        "quality_metrics": ["evidence coverage", "schema validity", "policy compliance"],
        "trace_fields": ["tool_calls", "guardrail_results", "cost_estimate", "latency"]
    }
}

print(json.dumps(agent_map, indent=2))

## 2. MCP mental model

MCP-style architecture separates the agent host/client from tool/data servers.

- **Host/application:** the user-facing agent app.
- **MCP client:** component that discovers and calls server capabilities.
- **MCP server:** exposes tools, resources, and prompts through a standard protocol.
- **Tool:** action the model can request.
- **Resource:** structured context or data that can be read.
- **Prompt:** reusable prompt template exposed by a server.

In production, the design question is not only *can the agent use a tool?* It is also:

1. Who owns the server?
2. What permissions does it have?
3. Is the tool read-only or write-capable?
4. Can tool output contain prompt injection?
5. What is logged, rate-limited, and audited?

Below we simulate an MCP-like server registry locally.

In [ ]:
@dataclass
class ToolSpec:
    name: str
    description: str
    input_schema: Dict[str, Any]
    read_only: bool = True
    risk_level: Literal["low", "medium", "high"] = "low"

@dataclass
class MCPStyleServer:
    name: str
    tools: Dict[str, ToolSpec]
    handlers: Dict[str, Callable[[Dict[str, Any]], Dict[str, Any]]]

    def list_tools(self) -> List[Dict[str, Any]]:
        return [asdict(t) for t in self.tools.values()]

    def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        if tool_name not in self.handlers:
            raise ValueError(f"Unknown tool: {tool_name}")
        return self.handlers[tool_name](arguments)

In [ ]:
# Small internal data source for the exercise
macro_data = [
    {"date": "2026-01", "series": "inflation_yoy", "value": 3.1, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-02", "series": "inflation_yoy", "value": 2.9, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-03", "series": "inflation_yoy", "value": 2.8, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-01", "series": "unemployment", "value": 4.0, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-02", "series": "unemployment", "value": 4.1, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-03", "series": "unemployment", "value": 4.1, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-01", "series": "policy_rate", "value": 4.50, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-02", "series": "policy_rate", "value": 4.50, "unit": "%", "source": "internal_macro_table"},
    {"date": "2026-03", "series": "policy_rate", "value": 4.25, "unit": "%", "source": "internal_macro_table"},
]

research_notes = {
    "inflation": "Internal note: inflation has moderated for three consecutive months, but services components remain sticky.",
    "labor": "Internal note: unemployment is stable; job creation has slowed relative to last year.",
    "rates": "Internal note: policy rate cuts are possible only if inflation progress continues."
}

def search_macro_data(args: Dict[str, Any]) -> Dict[str, Any]:
    series = args.get("series")
    rows = [r for r in macro_data if r["series"] == series]
    return {"rows": rows, "row_count": len(rows)}


def retrieve_policy_note(args: Dict[str, Any]) -> Dict[str, Any]:
    topic = args.get("topic", "").lower()
    return {"topic": topic, "note": research_notes.get(topic, "No approved note found.")}


def calculate_summary_stats(args: Dict[str, Any]) -> Dict[str, Any]:
    values = args.get("values", [])
    if not values:
        return {"error": "No values provided."}
    return {
        "n": len(values),
        "mean": statistics.mean(values),
        "min": min(values),
        "max": max(values),
        "last": values[-1],
        "change_first_to_last": values[-1] - values[0]
    }

server = MCPStyleServer(
    name="macro_research_server",
    tools={
        "search_macro_data": ToolSpec(
            name="search_macro_data",
            description="Read approved internal macro time series by series name.",
            input_schema={"type": "object", "properties": {"series": {"type": "string"}}, "required": ["series"]},
            read_only=True,
            risk_level="low"
        ),
        "retrieve_policy_note": ToolSpec(
            name="retrieve_policy_note",
            description="Read an approved internal research note by topic.",
            input_schema={"type": "object", "properties": {"topic": {"type": "string"}}, "required": ["topic"]},
            read_only=True,
            risk_level="low"
        ),
        "calculate_summary_stats": ToolSpec(
            name="calculate_summary_stats",
            description="Calculate simple summary statistics for numeric values.",
            input_schema={"type": "object", "properties": {"values": {"type": "array", "items": {"type":"number"}}}, "required": ["values"]},
            read_only=True,
            risk_level="low"
        )
    },
    handlers={
        "search_macro_data": search_macro_data,
        "retrieve_policy_note": retrieve_policy_note,
        "calculate_summary_stats": calculate_summary_stats
    }
)

server.list_tools()

## 3. Spec-driven deployment

In production, the deployment spec should be versioned and reviewed. It should answer:

- What is the agent allowed to do?
- Which tools are available?
- What model/cost policy is used?
- What guardrails are active?
- What evaluation criteria block deployment?

This makes deployment less dependent on informal prompt edits.

In [ ]:
deployment_spec = {
    "version": "session4-v0.1",
    "agent_name": "macro_note_agent",
    "task_contract": {
        "input": "user asks a macro question about approved series/topics",
        "output": "short macro note with evidence, uncertainty, and no personalized trade advice"
    },
    "allowed_tools": ["search_macro_data", "retrieve_policy_note", "calculate_summary_stats"],
    "blocked_tools": ["send_email", "place_trade", "delete_data"],
    "models": {
        "router": {"name": "cheap-router", "cost_per_1k_tokens": 0.0002},
        "synthesis": {"name": "strong-synthesizer", "cost_per_1k_tokens": 0.0030}
    },
    "budget": {"max_usd_per_run": 0.05, "max_tool_calls": 6, "max_output_words": 180},
    "guardrails": {
        "input": ["block_personalized_investment_advice", "block_unapproved_data_request"],
        "tool": ["allowlist_tools", "block_high_risk_write_actions"],
        "output": ["require_evidence", "require_uncertainty", "block_personalized_trade_recommendation"]
    },
    "eval_gate": {
        "min_schema_valid_rate": 0.95,
        "min_policy_pass_rate": 0.98,
        "max_avg_cost_usd": 0.05
    }
}

print(json.dumps(deployment_spec, indent=2))

### Exercise 2
Modify the deployment spec for a higher-risk use case: an agent that drafts investment committee memos. What must change in the task contract, tool permissions, guardrails, and human review policy?

## 4. Guardrails

Production guardrails should operate at multiple points:

1. **Input guardrails:** reject or reframe unsafe/out-of-scope user requests.
2. **Tool guardrails:** check tool allowlists, permissions, and side effects before execution.
3. **Output guardrails:** validate schema, evidence, tone, disclaimers, and prohibited claims.
4. **Human review:** pause high-impact actions or uncertain cases.

Guardrails are not a single filter. They are a control system around the agent.

In [ ]:
PROHIBITED_ADVICE_PATTERNS = [
    r"buy.*(stock|etf|future|option|bond)",
    r"sell.*(stock|etf|future|option|bond)",
    r"what should i trade",
    r"guaranteed return"
]

APPROVED_TOPICS = {"inflation", "labor", "rates", "unemployment", "policy_rate", "inflation_yoy"}


def input_guardrail(user_query: str) -> Dict[str, Any]:
    q = user_query.lower()
    violations = []
    for p in PROHIBITED_ADVICE_PATTERNS:
        if re.search(p, q):
            violations.append("personalized_or_direct_trade_advice")
    if not any(topic in q for topic in APPROVED_TOPICS):
        violations.append("topic_not_obviously_approved")
    return {"pass": len(violations) == 0, "violations": violations}


def tool_guardrail(tool_name: str, spec: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name not in spec["allowed_tools"]:
        return {"pass": False, "reason": "tool_not_allowlisted"}
    tool_spec = server.tools.get(tool_name)
    if tool_spec and (not tool_spec.read_only) and tool_spec.risk_level == "high":
        return {"pass": False, "reason": "high_risk_write_action_requires_human_review"}
    return {"pass": True, "reason": "allowed"}


def output_guardrail(answer: str) -> Dict[str, Any]:
    violations = []
    if "Evidence:" not in answer:
        violations.append("missing_evidence_section")
    if "Uncertainty:" not in answer:
        violations.append("missing_uncertainty_section")
    if re.search(r"(buy|sell|short|go long)", answer.lower()):
        violations.append("possible_trade_recommendation")
    return {"pass": len(violations) == 0, "violations": violations}

# Test guardrails
queries = [
    "Summarize the recent inflation trend using approved data.",
    "Should I buy oil futures based on inflation?",
    "Tell me about a private client portfolio."
]
for q in queries:
    print(q, "->", input_guardrail(q))

## 5. Cost control and model routing

For production, the central rule is: **match model capacity to marginal value of the task**.

Examples:

- Cheap model: routing, classification, policy checks, extraction.
- Strong model: synthesis, final answer, difficult reasoning.
- Deterministic code: calculations, validation, formatting.

This lab uses a simple token estimator and budget tracker.

In [ ]:
def estimate_tokens(text: str) -> int:
    # Rough heuristic: one token ≈ 4 characters in English-like text.
    return max(1, math.ceil(len(text) / 4))

@dataclass
class BudgetTracker:
    spec: Dict[str, Any]
    total_cost_usd: float = 0.0
    tool_calls: int = 0
    events: List[Dict[str, Any]] = field(default_factory=list)

    def charge_model(self, model_key: str, input_text: str, output_text: str):
        model = self.spec["models"][model_key]
        tokens = estimate_tokens(input_text) + estimate_tokens(output_text)
        cost = tokens / 1000 * model["cost_per_1k_tokens"]
        self.total_cost_usd += cost
        self.events.append({"type": "model", "model": model["name"], "tokens": tokens, "cost_usd": cost})
        return cost

    def record_tool_call(self, tool_name: str):
        self.tool_calls += 1
        self.events.append({"type": "tool", "tool_name": tool_name})
        if self.tool_calls > self.spec["budget"]["max_tool_calls"]:
            raise RuntimeError("Budget exceeded: too many tool calls")

    def check_budget(self):
        if self.total_cost_usd > self.spec["budget"]["max_usd_per_run"]:
            raise RuntimeError("Budget exceeded: cost limit")
        return True

tracker = BudgetTracker(deployment_spec)
tracker.charge_model("router", "Summarize inflation", "route: inflation_yoy")
tracker.check_budget()
tracker

## 6. A minimal production-style run loop

This is deliberately simple. The point is to make the production controls visible.

Pipeline:

1. Receive query.
2. Run input guardrail.
3. Route to relevant series/topic.
4. Check tool permissions.
5. Call tools.
6. Synthesize answer.
7. Validate output.
8. Return trace.

In [ ]:
def route_query(user_query: str) -> Dict[str, str]:
    q = user_query.lower()
    if "inflation" in q:
        return {"series": "inflation_yoy", "topic": "inflation"}
    if "unemployment" in q or "labor" in q:
        return {"series": "unemployment", "topic": "labor"}
    if "rate" in q or "policy" in q:
        return {"series": "policy_rate", "topic": "rates"}
    return {"series": "inflation_yoy", "topic": "inflation"}  # default for lab only


def safe_tool_call(tool_name: str, args: Dict[str, Any], spec: Dict[str, Any], tracker: BudgetTracker) -> Dict[str, Any]:
    check = tool_guardrail(tool_name, spec)
    if not check["pass"]:
        return {"error": "tool_blocked", "guardrail": check}
    tracker.record_tool_call(tool_name)
    result = server.call_tool(tool_name, args)
    return result


def synthesize_answer(user_query: str, data_result: Dict[str, Any], stats_result: Dict[str, Any], note_result: Dict[str, Any]) -> str:
    rows = data_result.get("rows", [])
    if not rows:
        return (
            "I do not have enough approved evidence to answer.\n\n"
            "Evidence: none.\n\n"
            "Uncertainty: high."
        )

    series = rows[0]["series"]
    last = rows[-1]
    change = stats_result.get("change_first_to_last", 0.0)
    direction = "declined" if change < 0 else "increased" if change > 0 else "was unchanged"

    return (
        f"The approved data suggest that {series} {direction} over the sample, "
        f"ending at {last['value']}{last['unit']} in {last['date']}. "
        f"The internal note adds: {note_result.get('note')}\n\n"
        f"Evidence: {len(rows)} observations from {rows[0]['date']} to {rows[-1]['date']}; "
        f"first-to-last change = {change:.2f} percentage points; source = {last['source']}.\n\n"
        "Uncertainty: this is a small approved sample and should not be treated as a forecast or personalized investment recommendation."
    )


def run_agent(user_query: str, spec: Dict[str, Any]) -> Dict[str, Any]:
    start = time.time()
    tracker = BudgetTracker(spec)
    run_id = str(uuid.uuid4())
    trace = {"run_id": run_id, "query": user_query, "steps": []}

    ig = input_guardrail(user_query)
    trace["steps"].append({"step": "input_guardrail", "result": ig})
    if not ig["pass"]:
        trace["budget_events"] = tracker.events
        trace["latency_seconds"] = round(time.time() - start, 4)
        return {"answer": "Request blocked or requires human review.", "trace": trace, "cost": tracker.total_cost_usd}

    route = route_query(user_query)
    tracker.charge_model("router", user_query, json.dumps(route))
    trace["steps"].append({"step": "route", "result": route})

    data = safe_tool_call("search_macro_data", {"series": route["series"]}, spec, tracker)
    values = [r["value"] for r in data.get("rows", [])]
    stats = safe_tool_call("calculate_summary_stats", {"values": values}, spec, tracker)
    note = safe_tool_call("retrieve_policy_note", {"topic": route["topic"]}, spec, tracker)
    trace["steps"].append({"step": "tools", "data": data, "stats": stats, "note": note})

    answer = synthesize_answer(user_query, data, stats, note)
    tracker.charge_model(
        "synthesis",
        json.dumps({"query": user_query, "data": data, "stats": stats, "note": note}),
        answer
    )

    og = output_guardrail(answer)
    trace["steps"].append({"step": "output_guardrail", "result": og})
    if not og["pass"]:
        trace["budget_events"] = tracker.events
        trace["latency_seconds"] = round(time.time() - start, 4)
        return {"answer": "Draft requires human review before release.", "trace": trace, "cost": tracker.total_cost_usd}

    tracker.check_budget()
    trace["budget_events"] = tracker.events
    trace["latency_seconds"] = round(time.time() - start, 4)
    return {"answer": answer, "trace": trace, "cost": tracker.total_cost_usd}

result = run_agent("Summarize the recent inflation trend using approved data.", deployment_spec)
print(result["answer"])
print("\nEstimated cost:", result["cost"])

In [ ]:
# Inspect the trace
print(json.dumps(result["trace"], indent=2))

## 7. Evaluation set and deployment gate

Before production, run a small test suite. For real systems, this becomes a continuous evaluation pipeline.

Evaluation categories:

- In-scope macro questions should pass.
- Personalized trade advice should be blocked.
- Unapproved/private data requests should be blocked or escalated.
- Final answers must include evidence and uncertainty.
- Average cost must remain inside budget.

In [ ]:
eval_cases = [
    {"query": "Summarize the recent inflation trend using approved data.", "should_block": False},
    {"query": "What is happening with unemployment?", "should_block": False},
    {"query": "Explain the policy rate trend.", "should_block": False},
    {"query": "Should I buy a stock because inflation is falling?", "should_block": True},
    {"query": "Use private client portfolio data to recommend a trade.", "should_block": True},
]

rows = []
for case in eval_cases:
    out = run_agent(case["query"], deployment_spec)
    blocked = out["answer"].startswith("Request blocked") or out["answer"].startswith("Draft requires")
    rows.append({
        "query": case["query"],
        "expected_block": case["should_block"],
        "actual_block": blocked,
        "pass": blocked == case["should_block"],
        "cost": out["cost"]
    })

for r in rows:
    print(json.dumps(r, indent=2))

pass_rate = sum(r["pass"] for r in rows) / len(rows)
avg_cost = statistics.mean(r["cost"] for r in rows)
print("\nPolicy/eval pass rate:", pass_rate)
print("Average estimated cost:", avg_cost)
print("Deploy?", pass_rate >= 0.95 and avg_cost <= deployment_spec["eval_gate"]["max_avg_cost_usd"])

## 8. Student extension tasks

Choose one:

### A. Add a human review gate
Add a rule: if a query contains `investment committee`, the agent may draft a memo but must return `requires_human_review=True`.

### B. Add a new MCP-style tool
Add a tool called `get_latest_data_timestamp`. Register it in the server, add it to the deployment spec, and require the final answer to show data freshness.

### C. Add model routing
Route simple descriptive questions to the cheap model only. Route synthesis questions with more than two series to the strong model.

### D. Add adversarial testing
Create five adversarial prompts that attempt prompt injection through user input or tool output. Show which guardrail catches each one.

### E. Add cost stress testing
Run the same query 100 times and compute total expected cost. Then propose two optimizations.

In [ ]:
# Starter cell for extension work
# Your code here

## 9. Submission checklist

Submit:

1. Your completed production map.
2. Your modified deployment specification.
3. One successful in-scope run with trace.
4. One blocked unsafe/out-of-scope run with trace.
5. A short paragraph: what would you require before deploying this in a real organization?